In [ ]:
!pip install -q transformers safetensors pandas tqdm

In [ ]:
import os
import re
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
FULL_DATASET_CSV = "/content/drive/MyDrive/dataset_with_generated_scripts_asr.csv"

XLMR_MODEL_DIR = "/content/drive/MyDrive/xlmr_dual_head_model"

OUTPUT_DIR = "/content/drive/MyDrive/mt5_hint_dataset_generation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = f"{OUTPUT_DIR}/full_dataset_with_xlmr_hints.csv"

In [ ]:
# ============================================================
# PATH CONFIG
# ============================================================

# Your saved trained model folder
XLMR_MODEL_DIR = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/best_model"

# Your input CSV file
# Must contain: expected_script, generated_script
INPUT_CSV_PATH = "/content/drive/MyDrive/dataset_with_generated_scripts_asr.csv"

# Label map paths used during training
# Change names if your files are named differently
ROOT_MAP_PATH = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/id2root.json"
SUFFIX_MAP_PATH = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/id2suffix.json"

# Final output CSV
FINAL_OUTPUT_CSV = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/final_xlmr_hints.csv"

# Optional debug output
DEBUG_OUTPUT_CSV = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/debug_xlmr_hints.csv"

# Same max length used during XLM-R training
MAX_LEN_XLMR = 128

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
print("Model folder files:")
print(os.listdir(XLMR_MODEL_DIR))

assert os.path.exists(INPUT_CSV_PATH), f"Input CSV not found: {INPUT_CSV_PATH}"
assert os.path.exists(ROOT_MAP_PATH), f"Root map not found: {ROOT_MAP_PATH}"
assert os.path.exists(SUFFIX_MAP_PATH), f"Suffix map not found: {SUFFIX_MAP_PATH}"

model_safetensor_path = os.path.join(XLMR_MODEL_DIR, "model.safetensors")
model_bin_path = os.path.join(XLMR_MODEL_DIR, "pytorch_model.bin")

assert os.path.exists(model_safetensor_path) or os.path.exists(model_bin_path), \
    "No model.safetensors or pytorch_model.bin found inside best_model folder"

Model folder files:
['tokenizer.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json']


In [ ]:
with open(ROOT_MAP_PATH, "r", encoding="utf-8") as f:
    id2root = json.load(f)

with open(SUFFIX_MAP_PATH, "r", encoding="utf-8") as f:
    id2suffix = json.load(f)

# Convert JSON string keys to int
id2root = {int(k): v for k, v in id2root.items()}
id2suffix = {int(k): v for k, v in id2suffix.items()}

num_root_labels = len(id2root)
num_suffix_labels = len(id2suffix)

print("Root labels:", num_root_labels)
print("Suffix labels:", num_suffix_labels)

print("Sample roots:", list(id2root.items())[:5])
print("Sample suffixes:", list(id2suffix.items())[:5])

Root labels: 796
Suffix labels: 62
Sample roots: [(0, 'Account'), (1, 'Adk'), (2, 'Akd'), (3, 'Alibaba'), (4, 'Application')]
Sample suffixes: [(0, 'அ'), (1, 'அயே'), (2, 'ஆ'), (3, 'ஆன'), (4, 'ஆம்')]


In [ ]:
def clean_token(tok):
    tok = str(tok).strip()
    tok = tok.strip(".,!?;:\"“”‘’()[]{}")
    return tok


def tokenize(text):
    tokens = []
    for tok in str(text).strip().split():
        cleaned = clean_token(tok)
        if cleaned:
            tokens.append(cleaned)
    return tokens


def is_english_word(token):
    return bool(re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", str(token)))


def is_tamil_text(text):
    return bool(re.search(r"[\u0B80-\u0BFF]", str(text)))


def is_mixed_token(token):
    token = str(token)

    if "-" not in token:
        return False

    parts = token.split("-", 1)

    if len(parts) != 2:
        return False

    left, right = parts[0].strip(), parts[1].strip()

    return is_english_word(left) and is_tamil_text(right)


def get_token_class(token):
    if is_mixed_token(token):
        return "MIX"
    elif is_english_word(token):
        return "EN"
    elif is_tamil_text(token):
        return "TA"
    else:
        return "OTHER"


def split_mixed_token(token):
    if "-" not in str(token):
        return None, None

    left, right = str(token).split("-", 1)
    return left.strip(), right.strip()


def extract_root_suffix(token, token_class):
    if token_class == "MIX":
        root, suffix = split_mixed_token(token)
        return root if root else "", suffix if suffix else ""

    elif token_class == "EN":
        return token, "NULL"

    return "", ""


def get_context(tokens, idx, window=2):
    left_tokens = tokens[max(0, idx - window):idx]
    right_tokens = tokens[idx + 1:idx + 1 + window]

    left_context = " ".join(left_tokens).strip()
    right_context = " ".join(right_tokens).strip()

    return left_context, right_context


def build_model_input(left_context, token, right_context, token_class, root, suffix):
    return (
        f"LEFT={left_context} "
        f"TOKEN={token} "
        f"RIGHT={right_context} "
        f"CLASS={token_class} "
        f"ROOT={root} "
        f"SUFFIX={suffix}"
    )

In [ ]:
class XLMRDualHeadModel(nn.Module):
    def __init__(self, model_name, num_root_labels, num_suffix_labels):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.root_classifier = nn.Linear(hidden_size, num_root_labels)
        self.suffix_classifier = nn.Linear(hidden_size, num_suffix_labels)

    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # XLM-R uses first token representation like CLS
        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = self.dropout(cls_repr)

        root_logits = self.root_classifier(cls_repr)
        suffix_logits = self.suffix_classifier(cls_repr)

        return {
            "root_logits": root_logits,
            "suffix_logits": suffix_logits
        }

In [ ]:
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_MODEL_DIR)

xlmr_model = XLMRDualHeadModel(
    model_name="xlm-roberta-large",
    num_root_labels=num_root_labels,
    num_suffix_labels=num_suffix_labels
)

# Load trained weights
if os.path.exists(model_safetensor_path):
    from safetensors.torch import load_file
    state_dict = load_file(model_safetensor_path)
    print("Loading model.safetensors")

else:
    state_dict = torch.load(model_bin_path, map_location="cpu")
    print("Loading pytorch_model.bin")

# Handle possible Trainer prefix issue
new_state_dict = {}

for key, value in state_dict.items():
    if key.startswith("model."):
        new_key = key.replace("model.", "", 1)
    else:
        new_key = key

    new_state_dict[new_key] = value

missing_keys, unexpected_keys = xlmr_model.load_state_dict(new_state_dict, strict=False)

print("Missing keys:", missing_keys[:10])
print("Unexpected keys:", unexpected_keys[:10])

xlmr_model.to(device)
xlmr_model.eval()

print("Loaded trained XLM-R dual-head model.")

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading model.safetensors
Missing keys: []
Unexpected keys: []
Loaded trained XLM-R dual-head model.


In [ ]:
def predict_token_with_confidence(tokens, idx):
    token = tokens[idx]
    token_class = get_token_class(token)

    # We only predict EN and MIX tokens
    if token_class not in ["EN", "MIX"]:
        return {
            "original_token": token,
            "token_class": token_class,
            "wrong_root": None,
            "wrong_suffix": None,
            "pred_root": None,
            "pred_suffix": None,
            "root_conf": None,
            "suffix_conf": None,
            "corrected_token": token
        }

    wrong_root, wrong_suffix = extract_root_suffix(token, token_class)
    left_context, right_context = get_context(tokens, idx, window=2)

    model_input = build_model_input(
        left_context=left_context,
        token=token,
        right_context=right_context,
        token_class=token_class,
        root=wrong_root,
        suffix=wrong_suffix
    )

    enc = xlmr_tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN_XLMR,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = xlmr_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        root_probs = F.softmax(outputs["root_logits"], dim=1)
        suffix_probs = F.softmax(outputs["suffix_logits"], dim=1)

        root_pred_id = torch.argmax(root_probs, dim=1).item()
        suffix_pred_id = torch.argmax(suffix_probs, dim=1).item()

        root_conf = root_probs[0, root_pred_id].item()
        suffix_conf = suffix_probs[0, suffix_pred_id].item()

    pred_root = id2root[root_pred_id]
    pred_suffix = id2suffix[suffix_pred_id]

    if token_class == "EN":
        corrected_token = pred_root

    elif token_class == "MIX":
        if pred_suffix in ["NULL", "", None]:
            corrected_token = pred_root
        else:
            corrected_token = f"{pred_root}-{pred_suffix}"

    else:
        corrected_token = token

    return {
        "original_token": token,
        "token_class": token_class,
        "wrong_root": wrong_root,
        "wrong_suffix": wrong_suffix,
        "pred_root": pred_root,
        "pred_suffix": pred_suffix,
        "root_conf": root_conf,
        "suffix_conf": suffix_conf,
        "corrected_token": corrected_token
    }

In [ ]:
ROOT_CONF_THRESH_MIX = 00
SUFFIX_CONF_THRESH_MIX = 0.0


def should_correct_token(pred_info):
    token_class = pred_info["token_class"]
    original_token = pred_info["original_token"]
    corrected_token = pred_info["corrected_token"]

    # Only apply hints for MIX tokens
    if token_class != "MIX":
        return False

    # No change needed
    if corrected_token == original_token:
        return False

    wrong_root = pred_info.get("wrong_root")
    pred_root = pred_info.get("pred_root")
    pred_suffix = pred_info.get("pred_suffix")

    # Safety rule 1:
    # Do not change English root.
    # Example: school-க்கு should not become college-க்கு
    if wrong_root != pred_root:
        return False

    # Safety rule 2:
    # Do not produce bad suffix output.
    if pred_suffix in ["NULL", "", None]:
        return False

    # Safety rule 3:
    # Apply only if confidence is high
    if (
        pred_info["root_conf"] >= ROOT_CONF_THRESH_MIX and
        pred_info["suffix_conf"] >= SUFFIX_CONF_THRESH_MIX
    ):
        return True

    return False

In [ ]:
def build_hints_for_sentence(generated_sentence):
    tokens = tokenize(generated_sentence)

    hints = []
    debug_rows = []

    for idx in range(len(tokens)):
        pred_info = predict_token_with_confidence(tokens, idx)

        apply_change = should_correct_token(pred_info)

        if apply_change:
            original_token = pred_info["original_token"]
            corrected_token = pred_info["corrected_token"]

            hints.append(f"{original_token}=>{corrected_token}")

        debug_rows.append({
            "token_index": idx,
            "original_token": pred_info["original_token"],
            "token_class": pred_info["token_class"],
            "wrong_root": pred_info.get("wrong_root"),
            "wrong_suffix": pred_info.get("wrong_suffix"),
            "pred_root": pred_info.get("pred_root"),
            "pred_suffix": pred_info.get("pred_suffix"),
            "root_conf": pred_info.get("root_conf"),
            "suffix_conf": pred_info.get("suffix_conf"),
            "corrected_token": pred_info.get("corrected_token"),
            "apply_hint": apply_change
        })

    if len(hints) == 0:
        hint_text = "Null"
    else:
        hint_text = " ; ".join(hints)

    return hint_text, debug_rows

In [ ]:
df = pd.read_csv(INPUT_CSV_PATH)

print("Columns:", df.columns.tolist())
print("Rows:", len(df))

required_cols = ["expected_script", "generated_script"]

for col in required_cols:
    assert col in df.columns, f"Missing required column: {col}"

df[required_cols].head()

Columns: ['audio_wav_path', 'expected_script', 'generated_script']
Rows: 25897


,expected_script,generated_script
0,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...
1,One week later என்னோட card-ல இருந்து five nine...,One week later என்னோட card-ல இருந்து five nine...
2,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்
3,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்
4,வீடெல்லாம் நல்லா தான் இருக்கு ஆனா paint colour...,வீடெல்லாம் நல்லாத்தான் இருக்கு ஆனா paint colou...


In [ ]:
final_rows = []
all_debug_rows = []

for row_idx, row in tqdm(df.iterrows(), total=len(df), desc="Generating XLM-R hints"):
    expected_script = str(row["expected_script"]).strip()
    generated_script = str(row["generated_script"]).strip()

    if generated_script == "" or generated_script.lower() == "nan":
        hint_text = "Null"
        debug_rows = []

    else:
        hint_text, debug_rows = build_hints_for_sentence(generated_script)

    final_rows.append({
        "expected_script": expected_script,
        "generated_script": generated_script,
        "Hint": hint_text
    })

    for d in debug_rows:
        d["row_index"] = row_idx
        d["expected_script"] = expected_script
        d["generated_script"] = generated_script
        all_debug_rows.append(d)

final_df = pd.DataFrame(final_rows)
debug_df = pd.DataFrame(all_debug_rows)

final_df.to_csv(FINAL_OUTPUT_CSV, index=False, encoding="utf-8-sig")
debug_df.to_csv(DEBUG_OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved final CSV:", FINAL_OUTPUT_CSV)
print("Saved debug CSV:", DEBUG_OUTPUT_CSV)

final_df.head(20)

Generating XLM-R hints: 100%|██████████| 25897/25897 [26:33<00:00, 16.25it/s]


Saved final CSV: /content/drive/MyDrive/xlmr_large_dual_head_model_try_2/final_xlmr_hints.csv
Saved debug CSV: /content/drive/MyDrive/xlmr_large_dual_head_model_try_2/debug_xlmr_hints.csv


,expected_script,generated_script,Hint
0,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,Null
1,One week later என்னோட card-ல இருந்து five nine...,One week later என்னோட card-ல இருந்து five nine...,Null
2,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Null
3,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,Null
4,வீடெல்லாம் நல்லா தான் இருக்கு ஆனா paint colour...,வீடெல்லாம் நல்லாத்தான் இருக்கு ஆனா paint colou...,Null
5,Bro online shopping பற்றி ஒரு video போடுங்க Sr...,Bro online shopping பற்றி ஒரு video போடுங்க Sr...,Null
6,அண்ணா amazon website open பண்ணி பாருங்க ship t...,அண்ணா amazon website open பண்ணி பாருங்க shift ...,Null
7,டேய் நான் வர late ஆகும் நீங்க எல்லாரும் போங்க,டேய் நான் வர late ஆகும் நீங்க எல்லாரும் போங்க,Null
8,எதும் நல்ல data package இருந்தா சொல்லுங்க தேவப...,எதும் நல்ல data package இருந்தா சொல்லுங்க தேவை...,Null
9,எல்லாதுலையும் first-ஆ வாரது முக்கியம் இல்ல கடை...,எல்லாத்துடையும் first-ஆ வாறது முக்கியம் இல்ல க...,Null


In [ ]:
total_rows = len(final_df)
hint_rows = (final_df["Hint"] != "Null").sum()
null_rows = (final_df["Hint"] == "Null").sum()

print("Total rows:", total_rows)
print("Rows with hints:", hint_rows)
print("Rows with Null:", null_rows)
print("Hint rate:", hint_rows / total_rows)

Total rows: 25897
Rows with hints: 1077
Rows with Null: 24820
Hint rate: 0.04158782870602772


# Anthoer approch golden

In [ ]:
import os
import re
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

In [ ]:
# ============================================================
# PATH CONFIG
# ============================================================

# Your original CSV file
# Must contain: expected_script, generated_script
INPUT_CSV_PATH =  "/content/drive/MyDrive/dataset_with_generated_scripts_asr.csv"

# Your trained XLM-R model folder
XLMR_MODEL_DIR = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/best_model"

# Label maps used during XLM-R training
ROOT_MAP_PATH = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/id2root.json"
SUFFIX_MAP_PATH = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/id2suffix.json"

# Output files
XLMR_HINT_OUTPUT_CSV = "/content/drive/MyDrive/xlmr_predicted_hints_for_exp_d.csv"
GOLD_HINT_OUTPUT_CSV = "/content/drive/MyDrive/gold_hints_for_exp_d.csv"
EXPERIMENT_D_OUTPUT_CSV = "/content/drive/MyDrive/mt5_experiment_D_mixed_hints.csv"
DEBUG_XLMR_OUTPUT_CSV = "/content/drive/MyDrive/debug_xlmr_hints_for_exp_d.csv"

# Same max length used for XLM-R training
MAX_LEN_XLMR = 128

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
assert os.path.exists(INPUT_CSV_PATH), f"Input CSV not found: {INPUT_CSV_PATH}"
assert os.path.exists(XLMR_MODEL_DIR), f"XLM-R model folder not found: {XLMR_MODEL_DIR}"
assert os.path.exists(ROOT_MAP_PATH), f"Root map not found: {ROOT_MAP_PATH}"
assert os.path.exists(SUFFIX_MAP_PATH), f"Suffix map not found: {SUFFIX_MAP_PATH}"

model_safetensor_path = os.path.join(XLMR_MODEL_DIR, "model.safetensors")
model_bin_path = os.path.join(XLMR_MODEL_DIR, "pytorch_model.bin")

assert os.path.exists(model_safetensor_path) or os.path.exists(model_bin_path), \
    "No model.safetensors or pytorch_model.bin found in XLMR_MODEL_DIR"

print("All paths are valid.")
print("Model folder files:")
print(os.listdir(XLMR_MODEL_DIR))

All paths are valid.
Model folder files:
['tokenizer.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json']


In [ ]:
df = pd.read_csv(INPUT_CSV_PATH)

print("Columns:", df.columns.tolist())
print("Rows:", len(df))

required_cols = ["expected_script", "generated_script"]

for col in required_cols:
    assert col in df.columns, f"Missing required column: {col}"

df = df.dropna(subset=["expected_script", "generated_script"]).reset_index(drop=True)

df["expected_script"] = df["expected_script"].astype(str).str.strip()
df["generated_script"] = df["generated_script"].astype(str).str.strip()

print("Rows after removing empty expected/generated:", len(df))

df[["generated_script", "expected_script"]].head()

Columns: ['audio_wav_path', 'expected_script', 'generated_script']
Rows: 25897
Rows after removing empty expected/generated: 25897


,generated_script,expected_script
0,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...
1,One week later என்னோட card-ல இருந்து five nine...,One week later என்னோட card-ல இருந்து five nine...
2,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்
3,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்
4,வீடெல்லாம் நல்லாத்தான் இருக்கு ஆனா paint colou...,வீடெல்லாம் நல்லா தான் இருக்கு ஆனா paint colour...


In [ ]:
with open(ROOT_MAP_PATH, "r", encoding="utf-8") as f:
    id2root = json.load(f)

with open(SUFFIX_MAP_PATH, "r", encoding="utf-8") as f:
    id2suffix = json.load(f)

id2root = {int(k): v for k, v in id2root.items()}
id2suffix = {int(k): v for k, v in id2suffix.items()}

num_root_labels = len(id2root)
num_suffix_labels = len(id2suffix)

print("Root labels:", num_root_labels)
print("Suffix labels:", num_suffix_labels)

print("Sample root labels:", list(id2root.items())[:10])
print("Sample suffix labels:", list(id2suffix.items())[:10])

Root labels: 796
Suffix labels: 62
Sample root labels: [(0, 'Account'), (1, 'Adk'), (2, 'Akd'), (3, 'Alibaba'), (4, 'Application'), (5, 'Atm'), (6, 'Auction'), (7, 'Background'), (8, 'Bag'), (9, 'Bank')]
Sample suffix labels: [(0, 'அ'), (1, 'அயே'), (2, 'ஆ'), (3, 'ஆன'), (4, 'ஆம்'), (5, 'ஆல'), (6, 'ஆவும்'), (7, 'ஆவே'), (8, 'இல'), (9, 'இல்')]


In [ ]:
def clean_token(tok):
    tok = str(tok).strip()
    tok = tok.strip(".,!?;:\"“”‘’()[]{}")
    return tok


def tokenize(text):
    tokens = []

    for tok in str(text).strip().split():
        cleaned = clean_token(tok)
        if cleaned:
            tokens.append(cleaned)

    return tokens


def is_english_word(token):
    return bool(re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", str(token)))


def is_tamil_text(text):
    return bool(re.search(r"[\u0B80-\u0BFF]", str(text)))


def is_mixed_token(token):
    token = str(token)

    if "-" not in token:
        return False

    left, right = token.split("-", 1)
    left = left.strip()
    right = right.strip()

    return is_english_word(left) and is_tamil_text(right)


def get_token_class(token):
    if is_mixed_token(token):
        return "MIX"
    elif is_english_word(token):
        return "EN"
    elif is_tamil_text(token):
        return "TA"
    else:
        return "OTHER"


def split_mixed_token(token):
    token = str(token)

    if "-" not in token:
        return None, None

    left, right = token.split("-", 1)
    return left.strip(), right.strip()


def extract_root_suffix(token, token_class):
    if token_class == "MIX":
        root, suffix = split_mixed_token(token)
        return root if root else "", suffix if suffix else ""

    elif token_class == "EN":
        return token, "NULL"

    return "", ""


def get_context(tokens, idx, window=2):
    left_tokens = tokens[max(0, idx - window):idx]
    right_tokens = tokens[idx + 1:idx + 1 + window]

    left_context = " ".join(left_tokens).strip()
    right_context = " ".join(right_tokens).strip()

    return left_context, right_context


def build_model_input(left_context, token, right_context, token_class, root, suffix):
    return (
        f"LEFT={left_context} "
        f"TOKEN={token} "
        f"RIGHT={right_context} "
        f"CLASS={token_class} "
        f"ROOT={root} "
        f"SUFFIX={suffix}"
    )

In [ ]:
class XLMRDualHeadModel(nn.Module):
    def __init__(self, model_name, num_root_labels, num_suffix_labels):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.root_classifier = nn.Linear(hidden_size, num_root_labels)
        self.suffix_classifier = nn.Linear(hidden_size, num_suffix_labels)

    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = self.dropout(cls_repr)

        root_logits = self.root_classifier(cls_repr)
        suffix_logits = self.suffix_classifier(cls_repr)

        return {
            "root_logits": root_logits,
            "suffix_logits": suffix_logits
        }

In [ ]:
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_MODEL_DIR)

xlmr_model = XLMRDualHeadModel(
    model_name="xlm-roberta-large",
    num_root_labels=num_root_labels,
    num_suffix_labels=num_suffix_labels
)

if os.path.exists(model_safetensor_path):
    from safetensors.torch import load_file
    state_dict = load_file(model_safetensor_path)
    print("Loaded model.safetensors")
else:
    state_dict = torch.load(model_bin_path, map_location="cpu")
    print("Loaded pytorch_model.bin")

# Fix possible Trainer prefix
new_state_dict = {}

for key, value in state_dict.items():
    if key.startswith("model."):
        new_key = key.replace("model.", "", 1)
    else:
        new_key = key

    new_state_dict[new_key] = value

missing_keys, unexpected_keys = xlmr_model.load_state_dict(new_state_dict, strict=False)

print("Missing keys count:", len(missing_keys))
print("Unexpected keys count:", len(unexpected_keys))

if len(missing_keys) > 0:
    print("First missing keys:", missing_keys[:10])

if len(unexpected_keys) > 0:
    print("First unexpected keys:", unexpected_keys[:10])

xlmr_model.to(device)
xlmr_model.eval()

print("XLM-R model ready.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model.safetensors
Missing keys count: 0
Unexpected keys count: 0
XLM-R model ready.


In [ ]:
def predict_token_with_confidence(tokens, idx):
    token = tokens[idx]
    token_class = get_token_class(token)

    if token_class not in ["EN", "MIX"]:
        return {
            "original_token": token,
            "token_class": token_class,
            "wrong_root": None,
            "wrong_suffix": None,
            "pred_root": None,
            "pred_suffix": None,
            "root_conf": None,
            "suffix_conf": None,
            "corrected_token": token
        }

    wrong_root, wrong_suffix = extract_root_suffix(token, token_class)
    left_context, right_context = get_context(tokens, idx, window=2)

    model_input = build_model_input(
        left_context=left_context,
        token=token,
        right_context=right_context,
        token_class=token_class,
        root=wrong_root,
        suffix=wrong_suffix
    )

    enc = xlmr_tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN_XLMR,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = xlmr_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        root_probs = F.softmax(outputs["root_logits"], dim=1)
        suffix_probs = F.softmax(outputs["suffix_logits"], dim=1)

        root_pred_id = torch.argmax(root_probs, dim=1).item()
        suffix_pred_id = torch.argmax(suffix_probs, dim=1).item()

        root_conf = root_probs[0, root_pred_id].item()
        suffix_conf = suffix_probs[0, suffix_pred_id].item()

    pred_root = id2root[root_pred_id]
    pred_suffix = id2suffix[suffix_pred_id]

    if token_class == "EN":
        corrected_token = pred_root

    elif token_class == "MIX":
        if pred_suffix in ["NULL", "", None]:
            corrected_token = pred_root
        else:
            corrected_token = f"{pred_root}-{pred_suffix}"

    else:
        corrected_token = token

    return {
        "original_token": token,
        "token_class": token_class,
        "wrong_root": wrong_root,
        "wrong_suffix": wrong_suffix,
        "pred_root": pred_root,
        "pred_suffix": pred_suffix,
        "root_conf": root_conf,
        "suffix_conf": suffix_conf,
        "corrected_token": corrected_token
    }

In [ ]:
ROOT_CONF_THRESH_MIX = 0.90
SUFFIX_CONF_THRESH_MIX = 0.70

In [ ]:
# ============================================================
# XLM-R hint thresholds
# ============================================================

ROOT_CONF_THRESH_MIX = 0.90
SUFFIX_CONF_THRESH_MIX = 0.70

# If you want more hints but more noise, use:
# ROOT_CONF_THRESH_MIX = 0.0
# SUFFIX_CONF_THRESH_MIX = 0.0

In [ ]:
def should_accept_xlmr_hint(pred_info):
    token_class = pred_info["token_class"]
    original_token = pred_info["original_token"]
    corrected_token = pred_info["corrected_token"]

    # Only MIX suffix hints
    if token_class != "MIX":
        return False

    # No change
    if corrected_token == original_token:
        return False

    wrong_root = pred_info.get("wrong_root")
    pred_root = pred_info.get("pred_root")
    pred_suffix = pred_info.get("pred_suffix")

    # Safety: do not change English root
    if wrong_root != pred_root:
        return False

    # Do not accept empty suffix
    if pred_suffix in ["NULL", "", None]:
        return False

    root_conf = pred_info.get("root_conf")
    suffix_conf = pred_info.get("suffix_conf")

    if root_conf is None or suffix_conf is None:
        return False

    if root_conf >= ROOT_CONF_THRESH_MIX and suffix_conf >= SUFFIX_CONF_THRESH_MIX:
        return True

    return False

In [ ]:
def build_xlmr_hint_for_sentence(generated_script):
    tokens = tokenize(generated_script)

    hints = []
    debug_rows = []

    for idx in range(len(tokens)):
        pred_info = predict_token_with_confidence(tokens, idx)

        apply_hint = should_accept_xlmr_hint(pred_info)

        if apply_hint:
            original_token = pred_info["original_token"]
            corrected_token = pred_info["corrected_token"]
            hints.append(f"{original_token}=>{corrected_token}")

        debug_rows.append({
            "token_index": idx,
            "original_token": pred_info["original_token"],
            "token_class": pred_info["token_class"],
            "wrong_root": pred_info.get("wrong_root"),
            "wrong_suffix": pred_info.get("wrong_suffix"),
            "pred_root": pred_info.get("pred_root"),
            "pred_suffix": pred_info.get("pred_suffix"),
            "root_conf": pred_info.get("root_conf"),
            "suffix_conf": pred_info.get("suffix_conf"),
            "corrected_token": pred_info.get("corrected_token"),
            "apply_xlmr_hint": apply_hint
        })

    if len(hints) == 0:
        return "Null", debug_rows

    return " ; ".join(hints), debug_rows

In [ ]:
xlmr_hint_rows = []
all_debug_rows = []

for row_idx, row in tqdm(df.iterrows(), total=len(df), desc="Generating XLM-R predicted hints"):
    generated_script = row["generated_script"]
    expected_script = row["expected_script"]

    if generated_script.strip() == "":
        xlmr_hint = "Null"
        debug_rows = []
    else:
        xlmr_hint, debug_rows = build_xlmr_hint_for_sentence(generated_script)

    xlmr_hint_rows.append({
        "row_index": row_idx,
        "generated_script": generated_script,
        "expected_script": expected_script,
        "xlmr_hint": xlmr_hint
    })

    for d in debug_rows:
        d["row_index"] = row_idx
        d["generated_script"] = generated_script
        d["expected_script"] = expected_script
        all_debug_rows.append(d)

xlmr_hint_df = pd.DataFrame(xlmr_hint_rows)
debug_xlmr_df = pd.DataFrame(all_debug_rows)

xlmr_hint_df.to_csv(XLMR_HINT_OUTPUT_CSV, index=False, encoding="utf-8-sig")
debug_xlmr_df.to_csv(DEBUG_XLMR_OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved XLM-R hint CSV:", XLMR_HINT_OUTPUT_CSV)
print("Saved XLM-R debug CSV:", DEBUG_XLMR_OUTPUT_CSV)

print("Total rows:", len(xlmr_hint_df))
print("Rows with XLM-R hints:", (xlmr_hint_df["xlmr_hint"] != "Null").sum())
print("Rows with Null:", (xlmr_hint_df["xlmr_hint"] == "Null").sum())
print("XLM-R hint rate:", (xlmr_hint_df["xlmr_hint"] != "Null").mean())

xlmr_hint_df.head()

Generating XLM-R predicted hints: 100%|██████████| 25897/25897 [27:55<00:00, 15.46it/s]


Saved XLM-R hint CSV: /content/drive/MyDrive/xlmr_predicted_hints_for_exp_d.csv
Saved XLM-R debug CSV: /content/drive/MyDrive/debug_xlmr_hints_for_exp_d.csv
Total rows: 25897
Rows with XLM-R hints: 621
Rows with Null: 25276
XLM-R hint rate: 0.023979611538015987


,row_index,generated_script,expected_script,xlmr_hint
0,0,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,Null
1,1,One week later என்னோட card-ல இருந்து five nine...,One week later என்னோட card-ல இருந்து five nine...,Null
2,2,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Null
3,3,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,Null
4,4,வீடெல்லாம் நல்லாத்தான் இருக்கு ஆனா paint colou...,வீடெல்லாம் நல்லா தான் இருக்கு ஆனா paint colour...,Null


In [ ]:
def build_gold_hint_simple(generated_script, expected_script):
    gen_tokens = tokenize(generated_script)
    exp_tokens = tokenize(expected_script)

    hints = []

    min_len = min(len(gen_tokens), len(exp_tokens))

    for i in range(min_len):
        gen_tok = gen_tokens[i]
        exp_tok = exp_tokens[i]

        if gen_tok == exp_tok:
            continue

        gen_cls = get_token_class(gen_tok)
        exp_cls = get_token_class(exp_tok)

        # Focus mainly on English / MIX correction hints
        if gen_cls in ["EN", "MIX"] or exp_cls in ["EN", "MIX"]:
            hints.append(f"{gen_tok}=>{exp_tok}")

    if len(hints) == 0:
        return "Null"

    return " ; ".join(hints)

In [ ]:
from difflib import SequenceMatcher

def build_gold_hint_aligned(generated_script, expected_script):
    gen_tokens = tokenize(generated_script)
    exp_tokens = tokenize(expected_script)

    matcher = SequenceMatcher(None, gen_tokens, exp_tokens)

    hints = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        # Same tokens, no hint needed
        if tag == "equal":
            continue

        gen_span = gen_tokens[i1:i2]
        exp_span = exp_tokens[j1:j2]

        # -------------------------------------------------
        # Case 1: one token replaced by one token
        # Example:
        # card-ல => card-க்கு
        # -------------------------------------------------
        if tag == "replace" and len(gen_span) == 1 and len(exp_span) == 1:
            gen_tok = gen_span[0]
            exp_tok = exp_span[0]

            gen_cls = get_token_class(gen_tok)
            exp_cls = get_token_class(exp_tok)

            # Focus on EN / MIX corrections
            if gen_cls in ["EN", "MIX"] or exp_cls in ["EN", "MIX"]:
                hints.append(f"{gen_tok}=>{exp_tok}")

        # -------------------------------------------------
        # Case 2: one generated token should be deleted
        # Example:
        # remove => DELETE
        # -------------------------------------------------
        elif tag == "delete" and len(gen_span) == 1:
            gen_tok = gen_span[0]
            gen_cls = get_token_class(gen_tok)

            if gen_cls in ["EN", "MIX"]:
                hints.append(f"{gen_tok}=>DELETE")

        # -------------------------------------------------
        # Case 3: one expected token should be inserted
        # Usually avoid this for hinting because there is
        # no source token to replace.
        # -------------------------------------------------
        elif tag == "insert":
            continue

        # -------------------------------------------------
        # Case 4: multi-token replace/delete
        # Avoid because it can create noisy hints.
        # -------------------------------------------------
        else:
            continue

    if len(hints) == 0:
        return "Null"

    return " ; ".join(hints)

In [ ]:
gold_hint_rows = []

for row_idx, row in tqdm(df.iterrows(), total=len(df), desc="Generating gold hints"):
    generated_script = row["generated_script"]
    expected_script = row["expected_script"]

    gold_hint = build_gold_hint_aligned(generated_script, expected_script)

    gold_hint_rows.append({
        "row_index": row_idx,
        "generated_script": generated_script,
        "expected_script": expected_script,
        "gold_hint": gold_hint
    })

gold_hint_df = pd.DataFrame(gold_hint_rows)
gold_hint_df.to_csv(GOLD_HINT_OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved gold hint CSV:", GOLD_HINT_OUTPUT_CSV)

print("Total rows:", len(gold_hint_df))
print("Rows with gold hints:", (gold_hint_df["gold_hint"] != "Null").sum())
print("Rows with Null:", (gold_hint_df["gold_hint"] == "Null").sum())
print("Gold hint rate:", (gold_hint_df["gold_hint"] != "Null").mean())

gold_hint_df.head()

Generating gold hints: 100%|██████████| 25897/25897 [00:02<00:00, 12233.40it/s]


Saved gold hint CSV: /content/drive/MyDrive/gold_hints_for_exp_d.csv
Total rows: 25897
Rows with gold hints: 3821
Rows with Null: 22076
Gold hint rate: 0.14754604780476502


,row_index,generated_script,expected_script,gold_hint
0,0,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,Null
1,1,One week later என்னோட card-ல இருந்து five nine...,One week later என்னோட card-ல இருந்து five nine...,Null
2,2,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Null
3,3,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,Null
4,4,வீடெல்லாம் நல்லாத்தான் இருக்கு ஆனா paint colou...,வீடெல்லாம் நல்லா தான் இருக்கு ஆனா paint colour...,Null


In [ ]:
pd.set_option("display.max_colwidth", 200)

gold_hint_df[gold_hint_df["gold_hint"] != "Null"][
    ["generated_script", "expected_script", "gold_hint"]
].head(30)

,generated_script,expected_script,gold_hint
10,உங்க worst xm experience என்ன,உங்க worst exam experience என்ன,xm=>exam
18,என்ன bro அது team number எண்டா,என்ன bro அது teen number எண்டா,team=>teen
20,Hari அண்ணா Sri Lanka-ல gold saving எப்பிடி பண்றதுனு video ஒன்னு போடுங்க,Hari அண்ணா Sri Lanka-ல Gold saving எப்படி பண்றதுன்னு video ஒன்னு போடுங்க,gold=>Gold
23,எனக்கு தெரிஞ்சு அந்த ரெண்டு team தான் இந்த player-க்கு பயங்கரமா beat பண்ணுவாங்க,எனக்கு தெரிஞ்சு அந்த ரெண்டு team-ம் தான் இந்த player-க்கு பயங்கரமா bid பண்ணுவாங்க,team=>team-ம் ; beat=>bid
24,நாங்களும் இனி youtuber marry camera கொண்டு திரியோணும் எண்டு நினைக்கிறன்,நாங்களும் இனி youtuber மாறி camera கொண்டு திரியோணும் எண்டு நினைக்கிறன்,marry=>மாறி
25,Phone number-அ எப்படி change பண்ணுவது,Phone number எப்படி change பண்ணுவது,number-அ=>number
32,அண்ணா India products எப்பிடி Sri Lanka-ல இருந்து order பண்றது,அண்ணா India Products எப்பிடி Sri Lanka-ல இருந்து order பண்ணுறது,products=>Products
59,Fuel station-ல one percentage add ஆகுது அது legal-ஆ,Fuel Station-ல one percentage add ஆகுது அது Legal-ஆ,station-ல=>Station-ல ; legal-ஆ=>Legal-ஆ
63,Taboard-ல control key-ஐயும் இதையும் சேர்த்து அமத்து,Keyboard-ல control key ஐயும் இதையும் சேர்த்து அமத்து,Taboard-ல=>Keyboard-ல
65,அடுத்த கிழமை அவதார் படம் வருது நான் theatre-க்கு போகப்போறன் நீயும் வாறியா,அடுத்த கிழமை Avatar படம் வருது நான் theater-க்கு போகபோறன் நீயும் வாறியா,அவதார்=>Avatar


In [ ]:
merged_df = df.copy()
merged_df["row_index"] = merged_df.index

merged_df = merged_df.merge(
    gold_hint_df[["row_index", "gold_hint"]],
    on="row_index",
    how="left"
)

merged_df = merged_df.merge(
    xlmr_hint_df[["row_index", "xlmr_hint"]],
    on="row_index",
    how="left"
)

merged_df["gold_hint"] = merged_df["gold_hint"].fillna("Null")
merged_df["xlmr_hint"] = merged_df["xlmr_hint"].fillna("Null")

print("Merged rows:", len(merged_df))

print("Gold hint rows:", (merged_df["gold_hint"] != "Null").sum())
print("XLM-R hint rows:", (merged_df["xlmr_hint"] != "Null").sum())

merged_df[["generated_script", "expected_script", "gold_hint", "xlmr_hint"]].head(20)

Merged rows: 25897
Gold hint rows: 3821
XLM-R hint rows: 621


,generated_script,expected_script,gold_hint,xlmr_hint
0,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,எனக்கு மட்டும் தான் தோணுதா இவர் எல்லா videos-ல...,Null,Null
1,One week later என்னோட card-ல இருந்து five nine...,One week later என்னோட card-ல இருந்து five nine...,Null,Null
2,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Lecture online எண்டா இந்த கிழமை வீட்ட போகலாம்,Null,Null
3,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,அம்மா சமையல் முடிச்சதும் kitchen clean பண்ணான்,Null,Null
4,வீடெல்லாம் நல்லாத்தான் இருக்கு ஆனா paint colou...,வீடெல்லாம் நல்லா தான் இருக்கு ஆனா paint colour...,Null,Null
5,Bro online shopping பற்றி ஒரு video போடுங்க Sr...,Bro online shopping பற்றி ஒரு video போடுங்க Sr...,Null,Null
6,அண்ணா amazon website open பண்ணி பாருங்க shift ...,அண்ணா amazon website open பண்ணி பாருங்க ship t...,Null,Null
7,டேய் நான் வர late ஆகும் நீங்க எல்லாரும் போங்க,டேய் நான் வர late ஆகும் நீங்க எல்லாரும் போங்க,Null,Null
8,எதும் நல்ல data package இருந்தா சொல்லுங்க தேவை...,எதும் நல்ல data package இருந்தா சொல்லுங்க தேவப...,Null,Null
9,எல்லாத்துடையும் first-ஆ வாறது முக்கியம் இல்ல க...,எல்லாதுலையும் first-ஆ வாரது முக்கியம் இல்ல கடை...,Null,Null


In [ ]:
mixed_rows = []

for _, row in tqdm(merged_df.iterrows(), total=len(merged_df), desc="Creating raw mixed rows"):
    generated_script = row["generated_script"]
    expected_script = row["expected_script"]
    gold_hint = str(row["gold_hint"]).strip()
    xlmr_hint = str(row["xlmr_hint"]).strip()

    if gold_hint in ["", "nan", "None", "null"]:
        gold_hint = "Null"

    if xlmr_hint in ["", "nan", "None", "null"]:
        xlmr_hint = "Null"

    # 1. Gold hint row
    if gold_hint != "Null":
        mixed_rows.append({
            "generated_script": generated_script,
            "expected_script": expected_script,
            "hint_text": gold_hint,
            "hint_type": "gold"
        })

    # 2. Null hint row
    mixed_rows.append({
        "generated_script": generated_script,
        "expected_script": expected_script,
        "hint_text": "Null",
        "hint_type": "null"
    })

    # 3. XLM-R predicted hint row
    if xlmr_hint != "Null":
        mixed_rows.append({
            "generated_script": generated_script,
            "expected_script": expected_script,
            "hint_text": xlmr_hint,
            "hint_type": "xlmr"
        })

mixed_raw_df = pd.DataFrame(mixed_rows)

print("Raw mixed rows:", len(mixed_raw_df))
print(mixed_raw_df["hint_type"].value_counts())
print(mixed_raw_df["hint_type"].value_counts(normalize=True))

Creating raw mixed rows: 100%|██████████| 25897/25897 [00:01<00:00, 23110.81it/s]

Raw mixed rows: 30339
hint_type
null    25897
gold     3821
xlmr      621
Name: count, dtype: int64
hint_type
null    0.853588
gold    0.125944
xlmr    0.020469
Name: proportion, dtype: float64


In [ ]:
RANDOM_SEED = 42

TARGET_GOLD_RATIO = 0.60
TARGET_NULL_RATIO = 0.30
TARGET_XLMR_RATIO = 0.10

gold_part = mixed_raw_df[mixed_raw_df["hint_type"] == "gold"].copy()
null_part = mixed_raw_df[mixed_raw_df["hint_type"] == "null"].copy()
xlmr_part = mixed_raw_df[mixed_raw_df["hint_type"] == "xlmr"].copy()

print("Available gold:", len(gold_part))
print("Available null:", len(null_part))
print("Available xlmr:", len(xlmr_part))

Available gold: 3821
Available null: 25897
Available xlmr: 621


In [ ]:
# Use available gold as the anchor
target_gold = len(gold_part)

if target_gold == 0:
    raise ValueError("No gold hints were created. Check build_gold_hint_simple function or your data.")

target_total = int(target_gold / TARGET_GOLD_RATIO)
target_null = int(target_total * TARGET_NULL_RATIO)
target_xlmr = int(target_total * TARGET_XLMR_RATIO)

print("Target total:", target_total)
print("Target gold:", target_gold)
print("Target null:", target_null)
print("Target xlmr:", target_xlmr)

# Sample null rows
if len(null_part) >= target_null:
    null_sample = null_part.sample(n=target_null, random_state=RANDOM_SEED)
else:
    null_sample = null_part.sample(n=target_null, replace=True, random_state=RANDOM_SEED)

# Sample XLM-R rows
if target_xlmr == 0 or len(xlmr_part) == 0:
    xlmr_sample = xlmr_part.head(0).copy()
    print("Warning: No XLM-R rows used.")
else:
    if len(xlmr_part) >= target_xlmr:
        xlmr_sample = xlmr_part.sample(n=target_xlmr, random_state=RANDOM_SEED)
    else:
        xlmr_sample = xlmr_part.sample(n=target_xlmr, replace=True, random_state=RANDOM_SEED)

exp_d_df = pd.concat(
    [gold_part, null_sample, xlmr_sample],
    ignore_index=True
)

exp_d_df = exp_d_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Final Experiment D rows:", len(exp_d_df))
print(exp_d_df["hint_type"].value_counts())
print(exp_d_df["hint_type"].value_counts(normalize=True))

Target total: 6368
Target gold: 3821
Target null: 1910
Target xlmr: 636
Final Experiment D rows: 6367
hint_type
gold    3821
null    1910
xlmr     636
Name: count, dtype: int64
hint_type
gold    0.600126
null    0.299984
xlmr    0.099890
Name: proportion, dtype: float64


In [ ]:
def build_mt5_input(row):
    return (
        "fix ASR with hints:\n"
        f"ASR: {row['generated_script']}\n"
        f"HINT: {row['hint_text']}"
    )


def build_mt5_target(row):
    return str(row["expected_script"]).strip()


exp_d_df["input_text"] = exp_d_df.apply(build_mt5_input, axis=1)
exp_d_df["target_text"] = exp_d_df.apply(build_mt5_target, axis=1)

exp_d_df[["hint_type", "input_text", "target_text"]].head(10)

,hint_type,input_text,target_text
0,xlmr,fix ASR with hints:\nASR: Technical wise-ஆ improvement வேணும் bro உங்க site-ல இருந்து\nHINT: wise-ஆ=>wise-அ,Technical wise-அ improvement வேணும் bro உங்க side-ல இருந்து
1,gold,fix ASR with hints:\nASR: நான் january water bottle order பண்ணினான் Korea fees எண்டு five hundred கேட்டாங்க but bottle worth just ninety eight\nHINT: Korea=>courier,நான் january water bottle order பண்ணினான் courier fees எண்டு five hundred கேட்டாங்க but bottle worth just ninety eight
2,gold,fix ASR with hints:\nASR: Class எடுத்தா higher studies easy-ஆ கிடைக்கும்\nHINT: easy-ஆ=>easy-அ,First class எடுத்தா higher studies easy-அ கிடைக்கும்
3,gold,fix ASR with hints:\nASR: Bro நான் last year driving exam pass பண்ணான் அப்போ temporary license sheet ஒன்னு தான் குடுத்தாங்க இந்த november அது expired ஆகுது\nHINT: driving=>drving,Bro நான் last year drving exam pass பண்ணான் அப்போ temporary license sheet ஒன்னு தான் குடுத்தாங்க இந்த november அது expired ஆகுது
4,xlmr,fix ASR with hints:\nASR: First class எடுத்தா higher studies easy-ஆ கிடைக்கும்\nHINT: easy-ஆ=>easy-அ,First class எடுத்தா higher studies easy-அ கிடைக்கும்
5,null,fix ASR with hints:\nASR: Train ticket waiting-ல இருக்கு confirm ஆகல\nHINT: Null,Train ticket waiting-ல இருக்கு confirm ஆகல
6,gold,fix ASR with hints:\nASR: Bro inverse video போடுங்களேன் please\nHINT: inverse=>invest,Bro invest video போடுங்களேன் please
7,gold,fix ASR with hints:\nASR: அண்ணா bcc savings account international transfer ஏற்றுக்கொள்ளுமா\nHINT: bcc=>boc,அண்ணா boc savings account international transfer-அ ஏற்றுகொள்ளுமா
8,gold,fix ASR with hints:\nASR: நான் India-ல இருந்து alibaba-ல பொருள் வாங்குறேன்\nHINT: India-ல=>இந்தியால,நான் இந்தியால இருந்து alibaba-ல பொருள் வாங்குறேன்
9,gold,fix ASR with hints:\nASR: Annunciation quality பாக்க superhits-அ வருவார் எல்லாரும் தயாராயிடுங்க\nHINT: Annunciation=>Production ; superhits-அ=>supervisor,Production quality பாக்க supervisor வருவார் எல்லாரும் தயாராயிருங்க


In [ ]:
exp_d_df.to_csv(EXPERIMENT_D_OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved Experiment D dataset:")
print(EXPERIMENT_D_OUTPUT_CSV)

print("Rows:", len(exp_d_df))
print(exp_d_df["hint_type"].value_counts())
print(exp_d_df["hint_type"].value_counts(normalize=True))

Saved Experiment D dataset:
/content/drive/MyDrive/mt5_experiment_D_mixed_hints.csv
Rows: 6367
hint_type
gold    3821
null    1910
xlmr     636
Name: count, dtype: int64
hint_type
gold    0.600126
null    0.299984
xlmr    0.099890
Name: proportion, dtype: float64


In [ ]:
print("Gold examples:")
display(exp_d_df[exp_d_df["hint_type"] == "gold"][["input_text", "target_text"]].head(5))

print("Null examples:")
display(exp_d_df[exp_d_df["hint_type"] == "null"][["input_text", "target_text"]].head(5))

print("XLM-R examples:")
display(exp_d_df[exp_d_df["hint_type"] == "xlmr"][["input_text", "target_text"]].head(5))

Gold examples:


,input_text,target_text
1,fix ASR with hints:\nASR: நான் january water bottle order பண்ணினான் Korea fees எண்டு five hundred கேட்டாங்க but bottle worth just ninety eight\nHINT: Korea=>courier,நான் january water bottle order பண்ணினான் courier fees எண்டு five hundred கேட்டாங்க but bottle worth just ninety eight
2,fix ASR with hints:\nASR: Class எடுத்தா higher studies easy-ஆ கிடைக்கும்\nHINT: easy-ஆ=>easy-அ,First class எடுத்தா higher studies easy-அ கிடைக்கும்
3,fix ASR with hints:\nASR: Bro நான் last year driving exam pass பண்ணான் அப்போ temporary license sheet ஒன்னு தான் குடுத்தாங்க இந்த november அது expired ஆகுது\nHINT: driving=>drving,Bro நான் last year drving exam pass பண்ணான் அப்போ temporary license sheet ஒன்னு தான் குடுத்தாங்க இந்த november அது expired ஆகுது
6,fix ASR with hints:\nASR: Bro inverse video போடுங்களேன் please\nHINT: inverse=>invest,Bro invest video போடுங்களேன் please
7,fix ASR with hints:\nASR: அண்ணா bcc savings account international transfer ஏற்றுக்கொள்ளுமா\nHINT: bcc=>boc,அண்ணா boc savings account international transfer-அ ஏற்றுகொள்ளுமா


Null examples:


,input_text,target_text
5,fix ASR with hints:\nASR: Train ticket waiting-ல இருக்கு confirm ஆகல\nHINT: Null,Train ticket waiting-ல இருக்கு confirm ஆகல
10,fix ASR with hints:\nASR: நீங்க advanced level exam எப்ப எடுத்தீங்க எனக்கு சொல்லவே இல்ல\nHINT: Null,நீங்க advanced level exam எப்ப எடுத்தீங்க எனக்கு சொல்லவே இல்ல
11,fix ASR with hints:\nASR: Powerbank கட்டில்ல வச்சனான் கீழ விழுந்துட்டு போல\nHINT: Null,Powerbank கட்டில்ல வச்சனான் கீழ விழுந்துட்டு போல
13,fix ASR with hints:\nASR: இந்த trailer கூட பாக்குற மாதிரி இல்லை இத தtheatre-ல போய் பாக்கணுமா\nHINT: Null,இந்த trailer கூட பார்க்கிற மாதிரி இல்லை இதை theatre-ல போய் பார்க்கனுமா
17,fix ASR with hints:\nASR: Hospital போக auto வர சொன்னான் ready-ஆ நில்லுங்கோ\nHINT: Null,Hospital போக auto வரசொன்னான் ready-ஆ நில்லுங்கோ


XLM-R examples:


,input_text,target_text
0,fix ASR with hints:\nASR: Technical wise-ஆ improvement வேணும் bro உங்க site-ல இருந்து\nHINT: wise-ஆ=>wise-அ,Technical wise-அ improvement வேணும் bro உங்க side-ல இருந்து
4,fix ASR with hints:\nASR: First class எடுத்தா higher studies easy-ஆ கிடைக்கும்\nHINT: easy-ஆ=>easy-அ,First class எடுத்தா higher studies easy-அ கிடைக்கும்
14,fix ASR with hints:\nASR: எங்க house-க்கு cleaning service-அ கூப்பிடணும்னு plan பண்ணிருக்கன்\nHINT: service-அ=>service-ஐ,எங்க house-க்கு cleaning service-ஐ கூப்பிடணும்னு plan பண்ணிருக்கன்
23,fix ASR with hints:\nASR: இல்லடா evening water board போய் supervisor-ஐ meet பண்ணோணும் இப்ப அங்க தான் போறன்\nHINT: supervisor-ஐ=>supervisor-அ,இல்லடா evening water board போய் supervisor-அ meet பண்ணோணும் இப்ப அங்கதான் போறன்
32,fix ASR with hints:\nASR: நீங்க support team-ஐ contact பண்ணினதையும் அதனு response-ஐயும் இதே comment-ல் சொல்லுங்க அது இன்னும் பெரிய help-ஆ இருக்கும் Sri Lankan creators-க்கு\nHINT: help-ஆ=>help-அ,நீங்க support team-ஐ contact பண்ணினதயும் அதுண்ர response-ஐயும் இதே comment-ல் சொல்லுங்க அது இன்னும் பெரிய help-அ இருக்கும் Sri Lankan creators-க்கு


In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_CSV = "/content/drive/MyDrive/mt5_experiment_D_train.csv"
VAL_CSV = "/content/drive/MyDrive/mt5_experiment_D_val.csv"

train_df, val_df = train_test_split(
    exp_d_df,
    test_size=0.1,
    random_state=42,
    shuffle=True,
    stratify=exp_d_df["hint_type"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_df.to_csv(TRAIN_CSV, index=False, encoding="utf-8-sig")
val_df.to_csv(VAL_CSV, index=False, encoding="utf-8-sig")

print("Train CSV:", TRAIN_CSV)
print("Val CSV:", VAL_CSV)

print("Train rows:", len(train_df))
print(train_df["hint_type"].value_counts(normalize=True))

print("Val rows:", len(val_df))
print(val_df["hint_type"].value_counts(normalize=True))

Train CSV: /content/drive/MyDrive/mt5_experiment_D_train.csv
Val CSV: /content/drive/MyDrive/mt5_experiment_D_val.csv
Train rows: 5730
hint_type
gold    0.600175
null    0.300000
xlmr    0.099825
Name: proportion, dtype: float64
Val rows: 637
hint_type
gold    0.599686
null    0.299843
xlmr    0.100471
Name: proportion, dtype: float64


# Train Mt5 with tokenize data set

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)



In [ ]:
!pip install jiwer evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.7 MB/s eta 0:00:00


In [ ]:
from jiwer import wer, cer

In [ ]:
# ============================================================
# PATH CONFIG
# ============================================================

# Experiment D train/validation CSV files
TRAIN_CSV = "/content/drive/MyDrive/mt5_experiment_D_train.csv"
VAL_CSV = "/content/drive/MyDrive/mt5_experiment_D_val.csv"

# Output directory
OUTPUT_DIR = "/content/drive/MyDrive/mt5_large_lora_r16_experiment_D"

# Base model
BASE_MODEL_NAME = "google/mt5-large"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /content/drive/MyDrive/mt5_large_lora_r16_experiment_D


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA memory allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
CUDA memory allocated: 0.0 GB


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))

print("Train columns:", train_df.columns.tolist())
print("Val columns:", val_df.columns.tolist())

required_cols = ["input_text", "target_text"]

for col in required_cols:
    assert col in train_df.columns, f"Missing train column: {col}"
    assert col in val_df.columns, f"Missing val column: {col}"

train_df = train_df.dropna(subset=["input_text", "target_text"]).reset_index(drop=True)
val_df = val_df.dropna(subset=["input_text", "target_text"]).reset_index(drop=True)

train_df["input_text"] = train_df["input_text"].astype(str).str.strip()
train_df["target_text"] = train_df["target_text"].astype(str).str.strip()

val_df["input_text"] = val_df["input_text"].astype(str).str.strip()
val_df["target_text"] = val_df["target_text"].astype(str).str.strip()

print("Train rows after cleaning:", len(train_df))
print("Val rows after cleaning:", len(val_df))

train_df[["input_text", "target_text"]].head()

Train rows: 5730
Val rows: 637
Train columns: ['generated_script', 'expected_script', 'hint_text', 'hint_type', 'input_text', 'target_text']
Val columns: ['generated_script', 'expected_script', 'hint_text', 'hint_type', 'input_text', 'target_text']
Train rows after cleaning: 5730
Val rows after cleaning: 637


,input_text,target_text
0,fix ASR with hints:\nASR: Bro mt five thirty ப...,Bro mt five பற்றி போடுங்க bro
1,fix ASR with hints:\nASR: உங்க lighting setup ...,உங்க lighting setup பற்றி சொல்லுங்க
2,fix ASR with hints:\nASR: Laptop keyboard-ல ke...,Laptop keyboard-ல keys sticky-ஆ இருக்கு அதான் ...
3,fix ASR with hints:\nASR: அந்த காணிகளுக்கு போய...,அந்த carnival-க்கு போய் magic show பாக்கோனும் ...
4,fix ASR with hints:\nASR: Tilly please exercis...,திலீப் please exercise பண்ணு


In [ ]:
if "hint_type" in train_df.columns:
    print("Train hint type distribution:")
    print(train_df["hint_type"].value_counts())
    print(train_df["hint_type"].value_counts(normalize=True))

if "hint_type" in val_df.columns:
    print("\nVal hint type distribution:")
    print(val_df["hint_type"].value_counts())
    print(val_df["hint_type"].value_counts(normalize=True))

Train hint type distribution:
hint_type
gold    3439
xlmr     572
Name: count, dtype: int64
hint_type
gold    0.857392
xlmr    0.142608
Name: proportion, dtype: float64

Val hint type distribution:
hint_type
gold    382
xlmr     64
Name: count, dtype: int64
hint_type
gold    0.856502
xlmr    0.143498
Name: proportion, dtype: float64


In [ ]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(val_df[["input_text", "target_text"]])

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 5730
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 637
})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("BF16:", USE_BF16)

print("Loaded:", BASE_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


BF16: True
Loaded: google/mt5-large


In [ ]:
!pip uninstall torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchao/*
Proceed (Y/n)? y
  Successfully uninstalled torchao-0.10.0


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "k", "v", "o"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,437,184 || all params: 1,751,247,872 || trainable%: 0.5389


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(val_df[["input_text", "target_text"]])

MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 256

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

Map:   0%|          | 0/5730 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

In [ ]:
def label_len(example):
    return {"label_len": len(example["labels"])}

train_check = tokenized_train.map(label_len)
val_check = tokenized_val.map(label_len)

print("Min train label length:", min(train_check["label_len"]))
print("Min val label length:", min(val_check["label_len"]))

bad_train = [i for i, x in enumerate(train_check["label_len"]) if x == 0]
bad_val = [i for i, x in enumerate(val_check["label_len"]) if x == 0]

print("Bad train labels:", len(bad_train))
print("Bad val labels:", len(bad_val))

Map:   0%|          | 0/5730 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

Min train label length: 5
Min val label length: 5
Bad train labels: 0
Bad val labels: 0


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

In [ ]:
import numpy as np
from jiwer import wer, cer

def safe_decode_token_ids(token_ids, tokenizer):
    token_ids = np.asarray(token_ids)

    if token_ids.ndim == 3:
        token_ids = np.argmax(token_ids, axis=-1)

    token_ids = np.where(token_ids < 0, tokenizer.pad_token_id, token_ids)
    token_ids = np.where(token_ids >= tokenizer.vocab_size, tokenizer.pad_token_id, token_ids)

    token_ids = token_ids.astype(np.int64)

    return tokenizer.batch_decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    labels = np.asarray(labels)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
    labels = np.where(labels >= tokenizer.vocab_size, tokenizer.pad_token_id, labels)
    labels = labels.astype(np.int64)

    decoded_preds = safe_decode_token_ids(preds, tokenizer)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    return {
        "wer": wer(decoded_labels, decoded_preds),
        "cer": cer(decoded_labels, decoded_preds)
    }

In [ ]:
#new

MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

Map:   0%|          | 0/5730 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    max_grad_norm=0.5,

    fp16=False,
    bf16=USE_BF16,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=1,

    logging_steps=25,
    report_to="none",

    load_best_model_at_end=False,

    gradient_checkpointing=True,
    remove_unused_columns=False
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
VAL_SUBSET_SIZE = 100

# Make sure subset size does not exceed validation size
VAL_SUBSET_SIZE = min(VAL_SUBSET_SIZE, len(tokenized_val))

# Shuffle and select subset
tokenized_val_small = tokenized_val.shuffle(seed=42).select(range(VAL_SUBSET_SIZE))

print("Full validation rows:", len(tokenized_val))
print("Small validation rows:", len(tokenized_val_small))

Full validation rows: 637
Small validation rows: 100


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val_small,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val_small,

    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

final_metrics = trainer.evaluate()

print("Final evaluation:")
print(final_metrics)

BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_lora_adapter")

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Saved best LoRA adapter to:")
print(BEST_MODEL_DIR)

PREDICTION_CSV = os.path.join(OUTPUT_DIR, "val_predictions_experiment_D.csv")

pred_output = trainer.predict(tokenized_val)

pred_ids = pred_output.predictions
label_ids = pred_output.label_ids

label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(
    pred_ids,
    skip_special_tokens=True
)

decoded_labels = tokenizer.batch_decode(
    label_ids,
    skip_special_tokens=True
)

decoded_preds = [x.strip() for x in decoded_preds]
decoded_labels = [x.strip() for x in decoded_labels]

pred_df = val_df.copy()
pred_df["prediction"] = decoded_preds
pred_df["reference"] = decoded_labels

pred_df.to_csv(PREDICTION_CSV, index=False, encoding="utf-8-sig")

print("Saved validation predictions:")
print(PREDICTION_CSV)

pred_df[["input_text", "reference", "prediction"]].head(20)


from peft import PeftModel

# Load fresh base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# Load LoRA adapter
inference_model = PeftModel.from_pretrained(
    base_model,
    BEST_MODEL_DIR
)

inference_model.to(device)
inference_model.eval()

inference_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)

print("Inference model loaded.")

def correct_asr_with_mt5(generated_script, hint_text="Null", max_new_tokens=256):
    input_text = (
        "fix ASR with hints:\n"
        f"ASR: {generated_script}\n"
        f"HINT: {hint_text}"
    )

    enc = inference_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        output_ids = inference_model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )

    prediction = inference_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return prediction.strip()

test_generated = "naan school-ல ponen"
test_hint = "school-ல=>school-க்கு"

prediction = correct_asr_with_mt5(test_generated, test_hint)

print("Generated:", test_generated)
print("Hint:", test_hint)
print("Prediction:", prediction)


Step,Training Loss,Validation Loss
500,194.423047,6.558570
1000,107.916621,3.689742


Final evaluation:
{'eval_loss': 3.687042236328125, 'eval_runtime': 16.2062, 'eval_samples_per_second': 6.17, 'eval_steps_per_second': 6.17, 'epoch': 3.0}
Saved best LoRA adapter to:
/content/drive/MyDrive/mt5_large_lora_r16_experiment_D/best_lora_adapter


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



KeyboardInterrupt: 

In [ ]:
print("Train rows:", len(train_df))
print("Val rows:", len(val_df))

print("Empty input_text train:", (train_df["input_text"].astype(str).str.strip() == "").sum())
print("Empty target_text train:", (train_df["target_text"].astype(str).str.strip() == "").sum())

print("Empty input_text val:", (val_df["input_text"].astype(str).str.strip() == "").sum())
print("Empty target_text val:", (val_df["target_text"].astype(str).str.strip() == "").sum())

train_df = train_df.dropna(subset=["input_text", "target_text"]).copy()
val_df = val_df.dropna(subset=["input_text", "target_text"]).copy()

train_df["input_text"] = train_df["input_text"].astype(str).str.strip()
train_df["target_text"] = train_df["target_text"].astype(str).str.strip()

val_df["input_text"] = val_df["input_text"].astype(str).str.strip()
val_df["target_text"] = val_df["target_text"].astype(str).str.strip()

train_df = train_df[
    (train_df["input_text"] != "") &
    (train_df["target_text"] != "") &
    (train_df["target_text"].str.lower() != "nan")
].reset_index(drop=True)

val_df = val_df[
    (val_df["input_text"] != "") &
    (val_df["target_text"] != "") &
    (val_df["target_text"].str.lower() != "nan")
].reset_index(drop=True)

print("Clean train rows:", len(train_df))
print("Clean val rows:", len(val_df))

Train rows: 5730
Val rows: 637
Empty input_text train: 0
Empty target_text train: 0
Empty input_text val: 0
Empty target_text val: 0
Clean train rows: 5730
Clean val rows: 637


In [ ]:
final_metrics = trainer.evaluate()

print("Final evaluation:")
print(final_metrics)

In [ ]:
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_lora_adapter")

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Saved best LoRA adapter to:")
print(BEST_MODEL_DIR)

In [ ]:
metrics_path = os.path.join(OUTPUT_DIR, "final_metrics.txt")

with open(metrics_path, "w", encoding="utf-8") as f:
    for key, value in final_metrics.items():
        f.write(f"{key}: {value}\n")

print("Saved metrics to:", metrics_path)

In [ ]:
PREDICTION_CSV = os.path.join(OUTPUT_DIR, "val_predictions_experiment_D.csv")

pred_output = trainer.predict(tokenized_val)

pred_ids = pred_output.predictions
label_ids = pred_output.label_ids

label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(
    pred_ids,
    skip_special_tokens=True
)

decoded_labels = tokenizer.batch_decode(
    label_ids,
    skip_special_tokens=True
)

decoded_preds = [x.strip() for x in decoded_preds]
decoded_labels = [x.strip() for x in decoded_labels]

pred_df = val_df.copy()
pred_df["prediction"] = decoded_preds
pred_df["reference"] = decoded_labels

pred_df.to_csv(PREDICTION_CSV, index=False, encoding="utf-8-sig")

print("Saved validation predictions:")
print(PREDICTION_CSV)

pred_df[["input_text", "reference", "prediction"]].head(20)

In [ ]:
from peft import PeftModel

# Load fresh base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# Load LoRA adapter
inference_model = PeftModel.from_pretrained(
    base_model,
    BEST_MODEL_DIR
)

inference_model.to(device)
inference_model.eval()

inference_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)

print("Inference model loaded.")

In [ ]:
def correct_asr_with_mt5(generated_script, hint_text="Null", max_new_tokens=256):
    input_text = (
        "fix ASR with hints:\n"
        f"ASR: {generated_script}\n"
        f"HINT: {hint_text}"
    )

    enc = inference_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        output_ids = inference_model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )

    prediction = inference_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return prediction.strip()

In [ ]:
test_generated = "naan school-ல ponen"
test_hint = "school-ல=>school-க்கு"

prediction = correct_asr_with_mt5(test_generated, test_hint)

print("Generated:", test_generated)
print("Hint:", test_hint)
print("Prediction:", prediction)

# load the new trained part mt5

In [ ]:
!pip install -q transformers peft accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

BASE_MODEL_NAME = "google/mt5-large"

BEST_MODEL_DIR = "/content/drive/MyDrive/mt5_large_lora_r16_experiment_D/best_lora_adapter"

# Load tokenizer saved with adapter
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)

# Load base mT5 model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

# Load LoRA adapter on top of base model
model = PeftModel.from_pretrained(
    base_model,
    BEST_MODEL_DIR
)

model.to(device)
model.eval()

print("LoRA mT5 model loaded successfully.")

Device: cuda


Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LoRA mT5 model loaded successfully.


In [ ]:
def correct_asr_with_mt5(generated_script, hint_text="Null"):
    input_text = (
        "fix ASR with hints:\n"
        f"ASR: {generated_script}\n"
        f"HINT: {hint_text}"
    )

    enc = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    bad_words_ids = tokenizer(
        ["<extra_id_0>", "<extra_id_1>", "<extra_id_2>", "ASR:", "HINT:"],
        add_special_tokens=False
    ).input_ids

    with torch.no_grad():
        output_ids = model.generate(
            **enc,
            max_new_tokens=64,
            num_beams=1,
            do_sample=False,
            repetition_penalty=1.5,
            no_repeat_ngram_size=3,
            bad_words_ids=bad_words_ids
        )

    prediction = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return prediction.strip()

In [ ]:
print(train_dataset[0]["input_text"])
print(train_dataset[0]["target_text"])

fix ASR with hints:
ASR: Bro mt five thirty போடுங்க bro
HINT: thirty=>பற்றி
Bro mt five பற்றி போடுங்க bro


In [ ]:
test_generated = "நான் school-ல போனேன்"
test_hint = "school-ல=>school-க்கு"

prediction = correct_asr_with_mt5(test_generated, test_hint)

print("Prediction:", prediction)

Prediction: αχ school-க்கு போனேன் நான் schoolல போனேன் HINT <extra_id_4> போனேன் <extra_id_5> போனேன் <extra_id_6> போனேன் <extra_id_7> போனேன் <extra_id_8> போனேன்=>school-க்கு=>school->schoolல=>schoolல <extra_id_9> போனேன் <extra_id_10> போனேன் school-ல=> schoolல=> <extra_id_11> போனேன் <extra_id_12> போனேன் <extra_id_13> போனேன் <extra_id_14>


In [ ]:
test_generated = "நான் school-ல போனேன்"
test_hint = "school-ல=>school-க்கு"

prediction = correct_asr_with_mt5(test_generated, test_hint)

print("Generated:", test_generated)
print("Hint:", test_hint)
print("Prediction:", prediction)

Generated: நான் school-ல போனேன்
Hint: school-ல=>school-க்கு
Prediction: <extra_id_0> போனேன் HINT: நான் school-ல போனேன் ASR: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான் school-ல போனேன் HINT:  <extra_id_18> school-ல போனேன் HINT: நான் school-ல போனேன் HINT: நான்


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL_NAME = "google/mt5-large"
BEST_MODEL_DIR = "/content/drive/MyDrive/mt5_large_lora_r16_experiment_D/best_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = PeftModel.from_pretrained(base_model, BEST_MODEL_DIR)
model.to(device)
model.eval()

print("Loaded correctly")

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded correctly


In [ ]:
def correct_asr_with_mt5(generated_script, hint_text="Null"):
    input_text = (
        "fix ASR with hints:\n"
        f"ASR: {generated_script}\n"
        f"HINT: {hint_text}"
    )

    enc = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    bad_tokens = [
        "ASR", "HINT", "HINT:", "=>",
        "<extra_id_0>", "<extra_id_1>", "<extra_id_2>",
        "<extra_id_3>", "<extra_id_4>", "<extra_id_5>",
        "<extra_id_6>", "<extra_id_7>", "<extra_id_8>",
        "<extra_id_9>", "<extra_id_10>"
    ]

    bad_words_ids = tokenizer(
        bad_tokens,
        add_special_tokens=False
    ).input_ids

    with torch.no_grad():
        output_ids = model.generate(
            **enc,
            max_new_tokens=40,
            num_beams=1,
            do_sample=False,
            repetition_penalty=2.0,
            no_repeat_ngram_size=2,
            bad_words_ids=bad_words_ids,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    prediction = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return prediction.strip()

In [ ]:
test_generated = "நான் school-ல போனேன்"
test_hint = "school-ல=>school-க்கு"

prediction = correct_asr_with_mt5(test_generated, test_hint)
print("Prediction:", prediction)

Prediction: αχ school-க்கு போனேன் நான் schoolல போனேன்னு HENT: school <extra_id_13> போனான்=>school <extra_id_14> போனோம் <extra_id_15> போனார் பள்ளிக்கு போ <extra_id_16>ன். <extra_id_17> போன <extra_id_18>ன்னார் <extra_id_47>


In [ ]:
prediction = correct_asr_with_mt5(
    "நான் school-ல போனேன்",
    "school-ல=>school-க்கு"
)

print(prediction)

αχ school-க்கு போனேன் நான் schoolல போனேன்னு HENT: school <extra_id_13> போனான்=>school <extra_id_14> போனோம் <extra_id_15> போனார் பள்ளிக்கு போ <extra_id_16>ன். <extra_id_17> போன <extra_id_18>ன்னார் <extra_id_47>


In [ ]:
print(train_dataset[0]["input_text"])
print(train_dataset[0]["target_text"])

fix ASR with hints:
ASR: Bro mt five thirty போடுங்க bro
HINT: thirty=>பற்றி
Bro mt five பற்றி போடுங்க bro


# New method

In [ ]:
!pip install -q transformers peft accelerate datasets jiwer sentencepiece

import os
import re
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from jiwer import wer, cer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from peft import LoraConfig, get_peft_model, PeftModel, TaskType

In [ ]:
TRAIN_CSV = "/content/drive/MyDrive/mt5_experiment_D_train.csv"
VAL_CSV = "/content/drive/MyDrive/mt5_experiment_D_val.csv"

OUTPUT_DIR = "/content/drive/MyDrive/mt5_large_lora_r16_experiment_D_fixed"
BASE_MODEL_NAME = "google/mt5-large"

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

print("Device:", device)
print("BF16:", USE_BF16)

Device: cuda
BF16: True


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

required_cols = ["input_text", "target_text"]

for col in required_cols:
    assert col in train_df.columns
    assert col in val_df.columns

train_df = train_df.dropna(subset=required_cols).reset_index(drop=True)
val_df = val_df.dropna(subset=required_cols).reset_index(drop=True)

for df in [train_df, val_df]:
    df["input_text"] = df["input_text"].astype(str).str.strip()
    df["target_text"] = df["target_text"].astype(str).str.strip()

# Remove empty rows
train_df = train_df[(train_df["input_text"] != "") & (train_df["target_text"] != "")]
val_df = val_df[(val_df["input_text"] != "") & (val_df["target_text"] != "")]

print("Train:", len(train_df))
print("Val:", len(val_df))

Train: 5730
Val: 637


In [ ]:
bad_train = train_df[
    train_df["target_text"].str.contains("ASR:|HINT:|=>|<extra_id", na=False)
]

bad_val = val_df[
    val_df["target_text"].str.contains("ASR:|HINT:|=>|<extra_id", na=False)
]

print("Bad train targets:", len(bad_train))
print("Bad val targets:", len(bad_val))

bad_train.head(10)

Bad train targets: 0
Bad val targets: 0


,generated_script,expected_script,hint_text,hint_type,input_text,target_text


In [ ]:
train_df = train_df[
    ~train_df["target_text"].str.contains("ASR:|HINT:|=>|<extra_id", na=False)
].reset_index(drop=True)

val_df = val_df[
    ~val_df["target_text"].str.contains("ASR:|HINT:|=>|<extra_id", na=False)
].reset_index(drop=True)

print("Clean train:", len(train_df))
print("Clean val:", len(val_df))

Clean train: 5730
Clean val: 637


In [ ]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(val_df[["input_text", "target_text"]])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "k", "v", "o"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,437,184 || all params: 1,751,247,872 || trainable%: 0.5389


In [ ]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 64

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

Map:   0%|          | 0/5730 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

In [ ]:
def safe_decode_token_ids(token_ids, tokenizer):
    token_ids = np.asarray(token_ids)

    # If predictions are logits: [batch, seq_len, vocab]
    if token_ids.ndim == 3:
        token_ids = np.argmax(token_ids, axis=-1)

    # Replace invalid ids
    token_ids = np.where(token_ids < 0, tokenizer.pad_token_id, token_ids)
    token_ids = np.where(token_ids >= tokenizer.vocab_size, tokenizer.pad_token_id, token_ids)

    token_ids = token_ids.astype(np.int64)

    return tokenizer.batch_decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    labels = np.asarray(labels)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
    labels = np.where(labels >= tokenizer.vocab_size, tokenizer.pad_token_id, labels)
    labels = labels.astype(np.int64)

    decoded_preds = safe_decode_token_ids(preds, tokenizer)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    return {
        "wer": wer(decoded_labels, decoded_preds),
        "cer": cer(decoded_labels, decoded_preds)
    }

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    max_grad_norm=1.0,

    fp16=False,
    bf16=USE_BF16,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=1,

    logging_steps=25,
    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    gradient_checkpointing=True,
    remove_unused_columns=False
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

final_metrics = trainer.evaluate()
print("Final evaluation:")
print(final_metrics)

BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_lora_adapter")

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Saved LoRA adapter to:")
print(BEST_MODEL_DIR)

Step,Training Loss,Validation Loss,Wer,Cer
500,18.162262,0.860117,0.207360,0.096932
1000,11.143728,0.580134,0.130042,0.040135
1500,11.232185,0.541922,0.120665,0.034886


Final evaluation:
{'eval_loss': 0.5419216156005859, 'eval_wer': 0.12066525123849965, 'eval_cer': 0.03488573621530615, 'eval_runtime': 917.4979, 'eval_samples_per_second': 0.694, 'eval_steps_per_second': 0.694, 'epoch': 5.0}
Saved LoRA adapter to:
/content/drive/MyDrive/mt5_large_lora_r16_experiment_D_fixed/best_lora_adapter


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32
)

inference_model = PeftModel.from_pretrained(
    base_model,
    BEST_MODEL_DIR
)

inference_model.to(device)
inference_model.eval()

print("Inference model loaded.")

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Inference model loaded.
